In [11]:
### 不带参数的函数装饰器
def funcDecorator(func):
    print(f'Decorating function {func.__name__}')
    def wrapper(*arg, **kwargs):
        ## 调用原函数前的逻辑
        print(f"Before calling {func.__name__}")
        ## 调用原函数
        result = func(*arg, **kwargs)
        ## 调用源函数后逻辑
        print(f"After calling {func.__name__}")
        return result
    return wrapper

In [14]:
@funcDecorator
def myFunc(a, b):
    print('My Function 执行中........')
    return a +  b

Decorating function myFunc


In [15]:
myFunc(1,1)

Before calling myFunc
My Function 执行中........
After calling myFunc


2

In [35]:
from functools import wraps

In [37]:
## 带参数的函数装饰器
def repeat(num_times):
    print(f'接受装饰器参数：{num_times}')
    ## 这个外层函数， 接收装饰器参数 /////
    def decorator_repeat(func):
        print(f'确认函数名称：{func.__name__}')
        ## ////这才是真正的装饰器/////
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f'num of times: {num_times},  Before {func.__name__} execute')
            for _ in range(num_times):
               result = func(*args, *kwargs)
            print(f'num of times: {num_times},  After {func.__name__} execute')
            return result
        return wrapper
    return decorator_repeat



In [40]:
repeat(num_times=2, func=funcDecorator)

TypeError: repeat() got an unexpected keyword argument 'func'

In [38]:
@repeat(num_times=4)
def greet(name):
    print(f'{greet.__name__} is running now.')
    print(f"Hello {name}")

接受装饰器参数：4
确认函数名称：greet


In [39]:
greet('ddma')

num of times: 4,  Before greet execute
greet is running now.
Hello ddma
greet is running now.
Hello ddma
greet is running now.
Hello ddma
greet is running now.
Hello ddma
num of times: 4,  After greet execute


### Function decorator的常见使用场景
1. 日志记录
2. 性能计时
3. 权限验证
4. 缓存/Memoization
5. 重试机制

In [4]:
### 日志记录
def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f'Calling {func.__name__} with args = {args}, kwargs={kwargs}')
        return func(*args, **kwargs)
    return wrapper

In [7]:
### 性能计时
import time
from functools import wraps

def time_it(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start
        print(f'{func.__name__} took {duration:.4f} seconds')
        return result
    return wrapper



In [8]:
@time_it
def logging_file():
    print('this is execution on file')
    
logging_file()

this is execution on file
logging_file took 0.0001 seconds


In [10]:
## 权限验证
def require_admin(func):
    @wraps(func)
    def wrapper(user, *args, **kwargs):
        if not user.is_admin:
            raise PermissionError('Admin rights required')
        return func(user, *args, **kwargs)
    return wrapper

In [13]:
from dataclasses import dataclass

@dataclass
class User:
    is_admin: bool
    name: str
    age: int

admin_user = User(is_admin=True, name='ddma', age=20)
non_admin_user = User(is_admin=False, name='ddma2', age=20)

In [17]:
@require_admin
def goto_landingpage(user, pagename):
    print(f'page name is {pagename}')

In [20]:
goto_landingpage(non_admin_user, "PAD operation daily report")
goto_landingpage(admin_user, "PAD operation daily report")

PermissionError: Admin rights required

In [25]:
### 缓存 / Memoization
from functools import lru_cache

@lru_cache(maxsize=32)
def expensive_calculation(x):
    return x * x

expensive_calculation(100)


10000

In [32]:
#### 重试机制, 重试机制是针对代码执行失败的尝试
import time
from functools import wraps

def retry(max_trials=3, delay=1):
    ## 带参数的装饰器
    print(f'设定的retry 次数{max_trials}')
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            trial = 0
            while trial < max_trials:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    trial += 1
                    if trial >= max_trials:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

In [47]:
@retry(max_trials=4, delay=2)
def myPrintln(x):
    print("Trying to connect...")
    raise ConnectionError("Server is down!")

设定的retry 次数4


In [48]:
myPrintln('This is my job！！！')

Trying to connect...
Trying to connect...
Trying to connect...
Trying to connect...


ConnectionError: Server is down!